# BoltzGen Training Job on SageMaker

이 노트북에서는 SageMaker Training Job을 사용하여 BoltzGen 모델을 Fine-tuning하는 방법을 실습합니다.

## 실습 순서

1. 환경 설정 및 라이브러리 임포트
2. Docker 이미지 URI 확인
3. 학습 데이터 준비 (S3 업로드)
4. SageMaker Training Job 생성 및 실행
5. 학습 모니터링
6. 학습 결과 확인

## 1. 환경 설정

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.pytorch import PyTorch
from sagemaker.inputs import TrainingInput
from sagemaker.debugger import TensorBoardOutputConfig

# SageMaker 세션 및 역할
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"Role: {role}")

In [ ]:
# S3 버킷 설정
S3_BUCKET = sagemaker_session.default_bucket()
S3_PREFIX = "boltzgen"

print(f"S3 Bucket: {S3_BUCKET}")
print(f"S3 Prefix: {S3_PREFIX}")

## 2. Docker 이미지 URI 확인

BoltzGen Docker 이미지가 ECR에 빌드되어 있어야 합니다.  
아직 빌드하지 않았다면 `scripts/build_and_push.sh`를 먼저 실행하세요.

```bash
cd session4 && bash scripts/build_and_push.sh
```

In [ ]:
# ECR 이미지 URI 설정
REPOSITORY_NAME = "boltzgen-sagemaker"
IMAGE_TAG = "latest"

# image_uri.txt 파일이 있으면 자동으로 읽기
image_uri_file = Path("scripts/image_uri.txt")
if image_uri_file.exists():
    IMAGE_URI = image_uri_file.read_text().strip()
else:
    IMAGE_URI = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{REPOSITORY_NAME}:{IMAGE_TAG}"

print(f"Image URI: {IMAGE_URI}")

## 3. 학습 데이터 준비

BoltzGen 학습에 필요한 데이터 구조:

```
s3://bucket/boltzgen/training-data/
├── targets/       # RCSB 처리된 타겟 구조
├── msa/           # 다중 서열 정렬(MSA) 데이터
└── mols/          # 화합물(CCD) 데이터
```

> **참고**: 전체 학습 데이터는 수십 GB 규모입니다.  
> 워크샵에서는 사전 준비된 데이터를 사용합니다.

In [ ]:
# 학습 데이터 S3 경로 설정
# 사전 준비된 데이터 경로를 사용하거나, 직접 업로드한 경로를 지정하세요
TRAINING_DATA_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/training-data/"

print(f"Training data S3 path: {TRAINING_DATA_S3}")

In [ ]:
# 워크샵용: 실제 학습 데이터를 소량만 다운로드하여 S3에 업로드
# 전체 데이터(수십 GB)를 받지 않고, 파이프라인 검증을 위해 최소한의 실제 데이터만 준비합니다.
import tarfile
import urllib.request
import tempfile

NUM_TARGET_FILES = 10  # 추출할 타겟 파일 수 (전체는 수만 개)

with tempfile.TemporaryDirectory() as tmpdir:
    local_data_dir = Path(tmpdir) / "training-data"
    
    # 1. CCD (화합물 데이터) 다운로드 → mols/ccd.pkl
    print("1/2) Downloading ccd.pkl...")
    mols_dir = local_data_dir / "mols"
    mols_dir.mkdir(parents=True)
    urllib.request.urlretrieve(
        "https://boltz1.s3.us-east-2.amazonaws.com/ccd.pkl",
        mols_dir / "ccd.pkl"
    )
    ccd_size = (mols_dir / "ccd.pkl").stat().st_size / 1e6
    print(f"   ccd.pkl downloaded ({ccd_size:.1f} MB)")
    
    # 2. RCSB targets tar에서 처음 N개 파일만 스트리밍 추출
    print(f"2/2) Streaming rcsb_processed_targets.tar (extracting first {NUM_TARGET_FILES} files)...")
    targets_dir = local_data_dir / "targets"
    targets_dir.mkdir(parents=True)
    
    url = "https://boltz1.s3.us-east-2.amazonaws.com/rcsb_processed_targets.tar"
    response = urllib.request.urlopen(url)
    
    count = 0
    with tarfile.open(fileobj=response, mode='r|') as tar:
        for member in tar:
            if member.isfile():
                tar.extract(member, path=targets_dir)
                count += 1
                print(f"   Extracted: {member.name}")
                if count >= NUM_TARGET_FILES:
                    break
    
    print(f"\n   Extracted {count} target files")
    
    # 3. S3에 업로드
    print(f"\nUploading to {TRAINING_DATA_S3}...")
    !aws s3 sync {local_data_dir} {TRAINING_DATA_S3}
    
print("Done! 소량의 실제 학습 데이터가 S3에 업로드되었습니다.")

In [ ]:
# (선택) 학습 데이터가 S3에 있는지 확인
!aws s3 ls {TRAINING_DATA_S3} --summarize

## 4. SageMaker Training Job 구성 및 실행

### 인스턴스별 권장 설정

| 인스턴스 | GPU | 메모리 | 배치 크기 | Gradient Accum | 비용 (시간당) |
|----------|-----|--------|-----------|----------------|---------------|
| ml.g5.2xlarge | 1 | 32GB | 1 | 16 | ~$1.63 |
| ml.g5.12xlarge | 4 | 96GB | 1 | 4 | ~$7.09 |
| ml.p4d.24xlarge | 8 | 320GB | 1 | 2 | ~$32.77 |

In [ ]:
# 학습 설정
INSTANCE_TYPE = "ml.g5.2xlarge"  # GPU 인스턴스 타입
INSTANCE_COUNT = 1               # 인스턴스 수
VOLUME_SIZE = 100                # EBS 볼륨 크기 (GB)
MAX_RUNTIME = 432000             # 최대 실행 시간 (초, 5일)
USE_SPOT = False                 # Spot 인스턴스 사용 여부

# 학습 하이퍼파라미터
CONFIG = "boltzgen_small"        # 학습 설정 (boltzgen_small / boltzgen / inverse_folding)
MAX_STEPS = 1000                 # 최대 학습 스텝 (워크샵에서는 짧게 설정)
SAVE_EVERY_N_STEPS = 500         # 체크포인트 저장 주기

In [ ]:
# 타임스탬프 및 Job 이름 생성
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_name = f"boltzgen-{CONFIG}-{timestamp}"
job_name = f"boltzgen-{timestamp}"

# 하이퍼파라미터 구성
hyperparameters = {
    'config': CONFIG,
    'name': experiment_name,
    'epochs': -1,
    'max-steps': MAX_STEPS,
    'save-every-n-steps': SAVE_EVERY_N_STEPS,
    'num-workers': 4,
}

# 환경 변수
environment = {
    'PYTHONUNBUFFERED': '1',
    'HF_HOME': '/tmp/huggingface',
}

print(f"Job name: {job_name}")
print(f"Experiment: {experiment_name}")
print(f"Hyperparameters: {json.dumps(hyperparameters, indent=2)}")

In [ ]:
# 출력 경로 설정
output_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/outputs/"
tensorboard_s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/tensorboard/"

# Spot 인스턴스용 체크포인트 설정
checkpoint_s3_uri = None
checkpoint_local_path = None
if USE_SPOT:
    checkpoint_s3_uri = f"s3://{S3_BUCKET}/{S3_PREFIX}/checkpoints/{job_name}/"
    checkpoint_local_path = '/opt/ml/checkpoints'

# TensorBoard 설정
tensorboard_output_config = TensorBoardOutputConfig(
    s3_output_path=tensorboard_s3_path,
    container_local_output_path='/opt/ml/output/tensorboard'
)

# CloudWatch 메트릭 정의
metric_definitions = [
    {'Name': 'train:loss', 'Regex': r'\[Metric\] train_loss=([0-9\.]+)'},
    {'Name': 'train:lr', 'Regex': r'\[Metric\] learning_rate=([0-9\.e\-]+)'},
    {'Name': 'train:epoch', 'Regex': r'\[Metric\] epoch=([0-9\.]+)'},
    {'Name': 'train:step', 'Regex': r'\[Metric\] step=([0-9]+)'},
    {'Name': 'val:loss', 'Regex': r'\[Metric\] val_loss=([0-9\.]+)'},
    {'Name': 'training:started', 'Regex': r'\[Metric\] training_started=([0-9]+)'},
    {'Name': 'training:completed', 'Regex': r'\[Metric\] training_completed=([0-9]+)'},
    {'Name': 'model:size_mb', 'Regex': r'\[Metric\] final_model_size_mb=([0-9\.]+)'},
]

print(f"Output path: {output_path}")
print(f"TensorBoard: {tensorboard_s3_path}")
if USE_SPOT:
    print(f"Spot checkpoint: {checkpoint_s3_uri}")

In [ ]:
# SageMaker PyTorch Estimator 생성
estimator = PyTorch(
    entry_point='train.py',
    source_dir='scripts',
    image_uri=IMAGE_URI,
    role=role,
    instance_count=INSTANCE_COUNT,
    instance_type=INSTANCE_TYPE,
    volume_size=VOLUME_SIZE,
    max_run=MAX_RUNTIME,
    hyperparameters=hyperparameters,
    environment=environment,
    output_path=output_path,
    base_job_name='boltzgen',
    use_spot_instances=USE_SPOT,
    max_wait=MAX_RUNTIME + 3600 if USE_SPOT else None,
    checkpoint_s3_uri=checkpoint_s3_uri,
    checkpoint_local_path=checkpoint_local_path,
    metric_definitions=metric_definitions,
    tensorboard_output_config=tensorboard_output_config,
    enable_sagemaker_metrics=True,
)

print("Estimator created successfully!")

In [ ]:
# 입력 채널 구성
inputs = {
    'training': TrainingInput(
        s3_data_type='S3Prefix',
        s3_data=TRAINING_DATA_S3,
        input_mode='File',
    ),
}

# (선택) 사전 학습된 체크포인트가 있는 경우
# PRETRAINED_CHECKPOINT_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/checkpoints/boltzgen1_diverse.ckpt"
# inputs['checkpoints'] = TrainingInput(
#     s3_data_type='S3Prefix',
#     s3_data=PRETRAINED_CHECKPOINT_S3,
#     input_mode='File',
# )

print("Input channels:")
for name, inp in inputs.items():
    print(f"  {name}: {inp.config['DataSource']['S3DataSource']['S3Uri']}")

In [ ]:
# Training Job 실행 (비동기)
estimator.fit(
    inputs=inputs,
    job_name=job_name,
    wait=False,  # 비동기 실행 (노트북이 블록되지 않음)
)

print(f"\nTraining Job launched: {job_name}")
print(f"Output: {output_path}{job_name}/")

## 5. 학습 모니터링

In [ ]:
# Training Job 상태 확인
sm_client = boto3.client('sagemaker')

response = sm_client.describe_training_job(TrainingJobName=job_name)
status = response['TrainingJobStatus']
print(f"Job Name: {job_name}")
print(f"Status: {status}")
print(f"Instance: {response['ResourceConfig']['InstanceType']}")

if status == 'Completed':
    print(f"\nTraining Time: {response['TrainingTimeInSeconds']}s")
    print(f"Billable Time: {response['BillableTimeInSeconds']}s")
elif status == 'Failed':
    print(f"\nFailure Reason: {response.get('FailureReason', 'N/A')}")

In [ ]:
# (선택) 학습 완료까지 대기
# 주의: 학습 시간이 길 수 있으므로 필요한 경우에만 실행하세요
# estimator.latest_training_job.wait(logs='All')

In [ ]:
# CloudWatch 로그 확인 (최근 로그)
!aws logs tail /aws/sagemaker/TrainingJobs --log-stream-name-prefix {job_name} --limit 20

## 6. 학습 결과 확인

In [ ]:
# 학습 완료 후 모델 아티팩트 다운로드
model_s3_path = f"{output_path}{job_name}/output/model.tar.gz"
print(f"Model artifact: {model_s3_path}")
print(f"\n모델 다운로드 명령어:")
print(f"  aws s3 cp {model_s3_path} ./")

In [ ]:
# TensorBoard 로그 확인
print(f"TensorBoard 로그 경로: {tensorboard_s3_path}{job_name}/")
print(f"\nTensorBoard 실행 방법:")
print(f"  aws s3 sync {tensorboard_s3_path}{job_name} ./tensorboard-logs/")
print(f"  tensorboard --logdir ./tensorboard-logs/")

In [ ]:
# S3에 저장된 학습 결과 목록 확인
!aws s3 ls {output_path}{job_name}/ --recursive --human-readable

## 정리

이 노트북에서는 다음을 학습했습니다:

- SageMaker `PyTorch` Estimator를 사용한 BoltzGen Training Job 구성
- Spot 인스턴스를 활용한 비용 최적화
- CloudWatch 메트릭 및 TensorBoard를 통한 학습 모니터링

다음 노트북 [2_inference_job.ipynb](./2_inference_job.ipynb)에서 학습된 모델을 사용한 추론을 실습합니다.